# Drishti M3: luggage detector (YOLO11n)

Fine-tunes YOLO11n on `guns-detection-cvwjs/luggagedataset-24pgo` (backpack / bag / trolley, 29,053 images; the authors' README states MIT). Goal: a bag detector that sees parked luggage on CCTV, where COCO weights found a bag in only about a quarter of ABODA frames (BENCH.md).

**Before running:** Settings -> Accelerator -> GPU T4, Internet on, and add a Kaggle secret `ROBOFLOW_API_KEY` (Add-ons -> Secrets).

In [ ]:
!pip -q install ultralytics==8.4.168 roboflow

In [ ]:
from kaggle_secrets import UserSecretsClient
from roboflow import Roboflow

rf = Roboflow(api_key=UserSecretsClient().get_secret('ROBOFLOW_API_KEY'))
ds = rf.workspace('guns-detection-cvwjs').project('luggagedataset-24pgo').version(1).download('yolov11')
print(ds.location)

In [ ]:
from ultralytics import YOLO

model = YOLO('yolo11n.pt')
model.train(data=f'{ds.location}/data.yaml', epochs=60, imgsz=640, batch=32, patience=15,
            project='runs', name='drishti', seed=0)

## Held-out metrics

Runs on the dataset's **test** split, which training never saw. Copy `metrics.json` into `docs/results/` in the repo.

In [ ]:
import json

best = YOLO('runs/drishti/weights/best.pt')
m = best.val(data=f'{ds.location}/data.yaml', split='test', imgsz=640)
metrics = {'dataset': 'guns-detection-cvwjs/luggagedataset-24pgo v1', 'split': 'test',
           'map50': float(m.box.map50), 'map50_95': float(m.box.map),
           'precision': float(m.box.mp), 'recall': float(m.box.mr),
           'classes': best.names, 'images': int(m.seen) if hasattr(m, 'seen') else None}
json.dump(metrics, open('luggage_metrics.json', 'w'), indent=1)
metrics

In [ ]:
import shutil
shutil.copy('runs/drishti/weights/best.pt', 'luggage.pt')
print('Download luggage.pt and luggage_metrics.json from the Output panel')

## Using the weights

The pipeline currently takes bags from the shared COCO detector. To use these weights, bench them first with `run_bench.py bag_detectors` (add the file to the `cands` dict), and wire them in as a second detector in `backend/app/detector.py` only if they beat COCO on the ABODA clips. Check the dataset version number on Roboflow Universe before running; version 1 is assumed here.